### **Create Bulk Analysis**

In [ ]:
from quinex_application.quinex import Quinex
quinex = Quinex()

In [ ]:
name = "quenix_application_query2"
response = quinex.intialize_analysis(name)
print(response)

### **Add Papers via DOI**

In [ ]:
from quinex_application.mongos_store import MongoStore
mongodb = MongoStore(quinex, database_name="quinex_db_query2", collection_name="papers_2025")

#### **Sort Paper by Year**

In [ ]:
from quinex_application.visualisation_metadata import VisualisationMetadata

visualisation = VisualisationMetadata("data/external/scopus_query2/scopus_2025_05_08_cleaned.csv")
df_doi = visualisation.df

In [ ]:
# Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2025 = dfs_by_year[2025]
print(len(df_2025))

In [ ]:
mongodb.delete_collection("papers")
mongodb.delete_collection("failed_papers")

In [ ]:
mongodb.insert_paper(df_2025, name)

In [ ]:
collections = mongodb.get_collection("papers_2025")

### **Successfully Added Papers**

In [ ]:
dois_df = df_2025['DOI'].dropna().unique().tolist()
print(len(dois_df))
existing_docs = mongodb.collection.find({"_id": {"$in": dois_df}}, {"_id": 1})
existing_dois = {doc["_id"] for doc in existing_docs}
display(df_doi[df_doi['DOI'].isin(existing_dois)])

### **Perform Bulk Analysis**

In [ ]:
dois = [doi for doi in df_2025["DOI"].head(100)]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois)

In [ ]:
display(df_2025.loc[df_2025['DOI'] == '10.1007/s40194-025-01955-1'])

In [ ]:
print(mongodb.get_collection("papers")[0]['analysis_result']['metadata']['bibliographic'].keys())
print(mongodb.get_collection("papers")[0]['analysis_result'].keys())
print(mongodb.get_collection("papers")[0]['analysis_result']['annotations'])